# 3DGS pipeline: images -> novel view video

Upload your multi-view photos as a Kaggle Dataset named `scene-images` (a zip or a folder is fine), then run this notebook cell by cell. It will: COLMAP SfM -> 3DGS train -> orbit render -> video.

In [ ]:
import os, sys, subprocess, json, shutil, glob
from pathlib import Path

ROOT = Path('/kaggle/working')
ROOT.mkdir(exist_ok=True)
print('ok')

## Step 1: system deps + COLMAP

In [ ]:
!apt-get update -qq
!apt-get install -y -qq colmap imagemagick ffmpeg > /dev/null 2>&1
!colmap version

## Step 2: clone + build 3DGS

In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git 2>&1 | tail -3
%cd gaussian-splatting
!pip install plyfile tqdm lpips torch torchvision --quiet
!pip install submodules/diff-gaussian-rasterization 2>&1 | tail -3
!pip install submodules/simple-knn 2>&1 | tail -3
print('3DGS build done')

## Step 3: gather images into data/scene/images

Handles a dataset uploaded as a zip (Kaggle may keep the nested folder) or a flat folder of images.

In [ ]:
kaggle_input = Path('/kaggle/input')

# locate the scene-images dataset, whatever its path
candidates = []
for p in kaggle_input.glob('scene-images/**'):
    candidates.append(p)
for p in kaggle_input.glob('**/scene-images*'):
    candidates.append(p)

found = next((p for p in candidates if p.is_dir()), None)
if found is None:
    print('searching all of /kaggle/input for jpg/png...')
    found = kaggle_input
imgs = sorted([p for p in found.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png')])
print(f'found {len(imgs)} images under {found}')

images = ROOT / 'data' / 'scene' / 'images'
images.mkdir(parents=True, exist_ok=True)
for i, src in enumerate(imgs):
    shutil.copy2(src, images)
print(f'copied {len(imgs)} images -> {images}')

## Step 4: COLMAP (features -> match -> SfM)

In [ ]:
%cd /kaggle/working
scene = ROOT / 'data' / 'scene'

# COLMAP ships a Qt/OpenGL GUI. In a headless container, run CPU-only SIFT
# (use_gpu 0) so it skips OpenGL/CUDA context creation entirely.
import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['DISPLAY'] = ''

# Downscale images to max 1600px and cap threads: full 5712px CPU SIFT blows up
# RAM and takes an hour. 1600px keeps plenty of features and runs fast.
!colmap feature_extractor \
  --database_path data/scene/database.db \
  --image_path data/scene/images \
  --SiftExtraction.use_gpu 0 \
  --SiftExtraction.max_num_features 8192 \
  --SiftExtraction.max_image_size 1600 \
  --SiftExtraction.estimate_affine_shape 0 \
  --SiftExtraction.num_threads 2

!colmap exhaustive_matcher \
  --database_path data/scene/database.db \
  --SiftMatching.use_gpu 0 \
  --SiftMatching.num_threads 2

!mkdir -p data/scene/sparse
!colmap mapper \
  --database_path data/scene/database.db \
  --image_path data/scene/images \
  --output_path data/scene/sparse


## Check: did COLMAP produce camera poses?

In [ ]:
from pathlib import Path
sp0 = ROOT / 'data' / 'scene' / 'sparse' / '0'
if (sp0 / 'images.bin').exists() or (sp0 / 'images.txt').exists():
    n = len(list(sp0.glob('images.*')))
    print('COLMAP SfM OK:', sp0)
else:
    print('COLMAP FAILED: no sparse/0 output. Check image overlap/texture.')
    print('sparse dir contents:', list((ROOT/'data'/'scene'/'sparse').rglob('*')))


## Step 5: train 3DGS

In [ ]:
%cd /kaggle/working/gaussian-splatting
!python train.py -s ../data/scene -m ../output/scene_run1 --iterations 7000 --test_iterations 7000 --save_iterations 7000

## Step 6: render orbit video (custom trajectory)

Uses render_custom.py (in this repo under scripts/). Copy it into the gaussian-splatting dir and run it; then ffmpeg joins the frames.

In [ ]:
%cd /kaggle/working/gaussian-splatting

# copy render_custom.py from the repo working dir (if the repo was added as input) or paste it inline
custom = Path('/kaggle/working/render_custom.py')
repo_script = Path('/kaggle/input/vision-worldmodel-projects/scripts/render_custom.py')
if repo_script.exists():
    shutil.copy2(repo_script, 'render_custom.py')
    print('copied render_custom.py from input repo')
elif custom.exists():
    print('using pre-copied render_custom.py')
else:
    print('render_custom.py not found - see README for the script body')

!python render_custom.py --model_path ../output/scene_run1 --output_dir ../output/orbit --frames 120

!ffmpeg -y -framerate 24 -pattern_type glob -i '../output/orbit/*.png' -c:v libx264 -pix_fmt yuv420p ../output/novel_view.mp4 2>&1 | tail -3
from pathlib import Path
v = Path('/kaggle/working/output/novel_view.mp4')
print('video:', v, 'size:', v.stat().st_size if v.exists() else 'MISSING')

## Step 7: metrics (PSNR on held-out COLMAP views)

Renders the COLMAP test split with the official render.py, then compares to ground truth.

In [ ]:
%cd /kaggle/working/gaussian-splatting
!python render.py -m ../output/scene_run1 --skip_train

# compute PSNR: gt in test/.../gt, prediction in test/.../renders
import numpy as np
from PIL import Image
gt_dir = Path('/kaggle/working/output/scene_run1/test')
renders = sorted(gt_dir.rglob('renders/*.png'))
gts = sorted(gt_dir.rglob('gt/*.png'))
def psnr(a, b):
    mse = np.mean((a.astype(float)-b.astype(float))**2)
    return 100 if mse == 0 else 20*np.log10(255.0/np.sqrt(mse))
scores = []
for r in renders:
    g = Path(str(r).replace('renders', 'gt'))
    if not g.exists():
        continue
    a = np.array(Image.open(g).convert('RGB'))
    b = np.array(Image.open(r).convert('RGB'))
    b = b[:a.shape[0], :a.shape[1]]
    scores.append(psnr(a, b))
print(f'PSNR: {np.mean(scores):.2f} +- {np.std(scores):.2f} dB over {len(scores)} views' if scores else 'no test views rendered (did you train without --eval?)')

## Step 8: package for download

In [ ]:
%cd /kaggle/working
!tar -czf output.tar.gz output/
!ls -lh output.tar.gz
print('Download output.tar.gz from the Kaggle output tab')